In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.optimize import minimize
from yahooquery import Ticker
import json
import os
from data_loader import get_or_fetch_historical_prices

USE_TRUE_HHI_OPTIMIZATION = False
AV_DB_FILE = "json\\new_etf_database.json" # 你的 Alpha Vantage 本地庫路徑

# ==========================================
# 信託底線權重 (Baseline Weights) 與 Alpha 融合
# ==========================================
# 系統信託底線的佔比 (0.0 = 完全聽從使用者, 1.0 = 完全使用系統底線, 0.5 = 各半)
ALPHA_BASELINE = 0.50 

# 專家先驗權重矩陣 (總和必須為 1.0)
BASELINE_WEIGHTS = {
    "Return_CAGR": 0.15,
    "Return_Div": 0.05,
    "Risk_Vol": 0.10,
    "Risk_MaxDD": 0.20,
    "Cost_ExpRatio": 0.20,
    "Liq_Volume": 0.05,
    "Liq_AUM": 0.05,
    "Div_Score": 0.15,
    "FinBERT_score": 0.05
}

# read from User_scenarios.txt (確保檔案存在)
try:
    with open("User_scenarios.txt", "r", encoding="utf-8") as f:
        case = f.read().strip()
except FileNotFoundError:
    case = "default_user"
# ==========================================
# 模組：建構 N x K 真實產業矩陣 (Sector Matrix S)
# ==========================================
def build_sector_matrix(etf_list, db_file):
    """
    將所有候選 ETF 的 JSON 產業分布，轉換為 N x K 的數學矩陣。
    N = ETF 數量, K = 所有出現過的獨特產業數量
    """
    if not os.path.exists(db_file):
        print("⚠️ 找不到 AV 資料庫，無法建立真實產業矩陣。")
        return None, []

    with open(db_file, 'r', encoding='utf-8') as f:
        db = json.load(f)

    # 1. 找出所有獨特的產業，建立產業字典 (Columns)
    all_sectors = set()
    for ticker in etf_list:
        if ticker in db and "Sector_Weights" in db[ticker]:
            all_sectors.update(db[ticker]["Sector_Weights"].keys())

    sector_list = list(all_sectors)
    K = len(sector_list)
    N = len(etf_list)
    
    if K == 0:
        return None, []

    # 2. 填入 N x K 權重矩陣
    S_matrix = np.zeros((N, K))
    for i, ticker in enumerate(etf_list):
        if ticker in db and "Sector_Weights" in db[ticker]:
            weights = db[ticker]["Sector_Weights"]
            for j, sector in enumerate(sector_list):
                S_matrix[i, j] = weights.get(sector, 0.0)
        else:
            # 防呆機制：若缺失該檔 ETF 的產業資料，強制將其視為 100% 未知產業
            # 在最佳化時會對其施加最嚴厲的集中度懲罰
            pass 

    print(f"📊 成功建立真實產業矩陣 (維度: {N} 檔 ETF x {K} 個產業)")
    return S_matrix, sector_list

def plot_portfolio_analytics_and_mpt(returns_matrix, optimal_weights, max_sharpe_weights, tickers):
    """視覺化模組：繪製歷史軌跡與 MPT 效率前緣 (結合精確解析解)"""
    print("\n啟動視覺化模組：計算歷史軌跡與 MPT 效率前緣...")
    
    sns.set_theme(style="whitegrid")
    plt.rcParams['font.sans-serif'] = ['Microsoft JhengHei', 'PingFang HK', 'Arial Unicode MS']
    plt.rcParams['axes.unicode_minus'] = False
    
    # 建立輸出資料夾
    os.makedirs("png", exist_ok=True)
    
    # --- 圖表 1：歷史淨值曲線與最大回撤 ---
    port_daily_returns = returns_matrix.dot(optimal_weights)
    cumulative_returns = (1 + port_daily_returns).cumprod()
    peak = cumulative_returns.cummax()
    drawdown = (cumulative_returns - peak) / peak
    
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), gridspec_kw={'height_ratios': [3, 1]}, sharex=True)
    
    ax1.plot(cumulative_returns.index, cumulative_returns, color='navy', linewidth=2)
    ax1.set_title('投資組合歷史淨值曲線 (Cumulative Returns)', fontsize=14, fontweight='bold')
    ax1.set_ylabel('累積淨值 (基期=1)', fontsize=12)
    
    ax2.fill_between(drawdown.index, drawdown * 100, 0, color='red', alpha=0.5)
    ax2.plot(drawdown.index, drawdown * 100, color='darkred', linewidth=1)
    ax2.set_title('歷史回撤幅度 (Drawdown %)', fontsize=14, fontweight='bold')
    ax2.set_ylabel('回撤 (%)', fontsize=12)
    ax2.set_xlabel('日期', fontsize=12)
    
    plt.tight_layout()
    plt.savefig(f'png\\{case}_portfolio_performance.png', dpi=300)
    plt.close()
    print(f"✅ 產出圖表：png\\{case}_portfolio_performance.png")

    # --- 圖表 2：蒙地卡羅模擬 MPT 效率前緣 ---
    annual_returns = returns_matrix.mean() * 252
    cov_matrix = returns_matrix.cov() * 252
    rf_rate = 0.04  # 假設無風險利率 4%
    
    # 1. 計算偏好驅動組合落點
    port_vol = np.sqrt(np.dot(optimal_weights.T, np.dot(cov_matrix, optimal_weights)))
    port_ret = np.dot(optimal_weights.T, annual_returns)
    
    # 2. 🚨 新增：計算精確的 Max Sharpe 組合落點
    exact_ms_vol = np.sqrt(np.dot(max_sharpe_weights.T, np.dot(cov_matrix, max_sharpe_weights)))
    exact_ms_ret = np.dot(max_sharpe_weights.T, annual_returns)
    ms_sharpe_ratio = (exact_ms_ret - rf_rate) / exact_ms_vol if exact_ms_vol > 0 else 0
    
    # 3. 改良版蒙地卡羅模擬 (加入稀疏性以拓展邊界)
    num_portfolios = 10000
    results = np.zeros((3, num_portfolios))
    
    for i in range(num_portfolios):
        weights = np.random.random(len(tickers))
        
        # 🚨 隨機將 50%~80% 的資產權重歸零，強迫模擬極端集中的組合
        if i > num_portfolios // 3: # 1/3傳統均勻分佈，剩下2/3做極端測試
            mask = np.random.rand(len(tickers)) > (np.random.uniform(0.2, 0.5))
            weights[mask] = 0
            
        if np.sum(weights) == 0: 
            weights[0] = 1 # 防呆
            
        weights /= np.sum(weights)
        p_ret = np.dot(weights, annual_returns)
        p_vol = np.sqrt(np.dot(weights.T, np.dot(cov_matrix, weights)))
        results[0,i] = p_vol
        results[1,i] = p_ret
        results[2,i] = (p_ret - 0.04) / p_vol
        
    plt.figure(figsize=(10, 7))
    scatter = plt.scatter(results[0,:] * 100, results[1,:] * 100, c=results[2,:], cmap='viridis', marker='o', s=10, alpha=0.3)
    plt.colorbar(scatter, label='夏普指標 (Sharpe Ratio)')
    
    # 標示專屬客製化組合 (紅色星星)
    plt.scatter(port_vol * 100, port_ret * 100, color='red', marker='*', s=300, edgecolor='black', 
                label=f'偏好驅動組合\n(報酬: {port_ret*100:.1f}%, 風險: {port_vol*100:.1f}%)', zorder=5)
    
    # 🚨 修正：標示全局最大夏普組合 (使用 SLSQP 算出的解析解，不再依賴蒙地卡羅)
    plt.scatter(exact_ms_vol * 100, exact_ms_ret * 100, color='blue', marker='X', s=150, edgecolor='black',
                label=f'傳統 Max Sharpe 組合\n(報酬: {exact_ms_ret*100:.1f}%, 風險: {exact_ms_vol*100:.1f}%)', zorder=5)

   # ==========================================
    # 🚨 新增：繪製資本市場線 (CML / Tangent Line)
    # ==========================================
    max_vol_plot = np.max(results[0,:]) * 100 * 1.05 # 將線條延伸至點雲最右側
    cml_x = np.array([0, exact_ms_vol * 100, max_vol_plot])
    # 直線方程式: y = R_f + Sharpe * x
    cml_y = rf_rate * 100 + ms_sharpe_ratio * cml_x

    plt.plot(cml_x, cml_y, color='darkorange', linestyle='--', linewidth=2, 
             label=f'資本市場線 (CML, Rf={rf_rate*100:.0f}%)', zorder=4)
             
    # 標示 Y 軸上的無風險利率起點
    plt.scatter(0, rf_rate * 100, color='darkorange', marker='D', s=80, edgecolor='black', zorder=5)

    # 強制 X 軸從 0 開始，以顯示完整的切線軌跡
    plt.xlim(left=0, right=max_vol_plot)

    plt.title('現代投資組合理論 (MPT) 效率前緣與資本市場線', fontsize=16, fontweight='bold', pad=15)
    plt.xlabel('年化波動率 (風險) %', fontsize=12)
    plt.ylabel('預期年化報酬率 %', fontsize=12)
    plt.legend(loc='lower right', frameon=True, shadow=True) # 圖例移至右下避免擋住 Y 截距
    
    plt.tight_layout()
    plt.savefig(f"png\\{case}_mpt_efficient_frontier.png", dpi=300)
    plt.close()
    print(f"✅ 產出圖表：png\\{case}_mpt_efficient_frontier.png")


def run_stage3_pipeline():
    print("啟動 Stage 3: 偏好驅動二次規劃與投資組合深度分析...")
    
    try:
        df_stage2 = pd.read_csv("csv\\stage2_final_user_universe.csv")
        df_stage0 = pd.read_csv("csv\\stage0_final_matrix.csv")
        df_scaled_features = pd.read_csv("csv\\stage2_normalized_features.csv")
        with open("json\\stage2_ahp_global_weights.json", "r", encoding="utf-8") as f:
            # 🚨 修正：提取 9 個全局權重
            global_weights = json.load(f)["Global_Weights"]
    except FileNotFoundError:
        print("❌ 找不到必要檔案，請確認 csv 檔案與 AHP 權重檔案存在。")
        return
    
    tickers = df_stage2['ETF'].tolist()
    
    # 抓取歷史價格並對齊資料
    print(f"⏳ 載入 {len(tickers)} 檔 ETF 進行最佳化與歷史回測...")
    price_matrix = get_or_fetch_historical_prices(tickers)
    returns_matrix = price_matrix.pct_change().dropna(how='all').ffill().dropna(axis=1)
    
    # 對齊有效 Tickers
    valid_tickers = [t for t in tickers if t in returns_matrix.columns]
    returns_matrix = returns_matrix[valid_tickers]
    n_assets = len(valid_tickers)
    
    df_merged = df_stage2[['ETF', 'User_Pref_Score']].merge(df_stage0, on='ETF', how='left')
    df_merged_valid = df_merged.set_index('ETF').loc[valid_tickers].reset_index()

    sector_matrix, sector_names = build_sector_matrix(valid_tickers, AV_DB_FILE)

    # 提取對應有效標的的正規化特徵
    df_scaled_valid = df_scaled_features.set_index('ETF').loc[valid_tickers].reset_index()

    # --- 輸出共變異數矩陣 CSV ---
    cov_matrix_annual = returns_matrix.cov() * 252
    #cov_matrix_annual.to_csv(f"csv\\{case}_covariance_matrix.csv")
    #print(f"✅ 產出共變異數矩陣檔案：csv\\{case}_covariance_matrix.csv")

    # --- 目標函數參數準備 ---
    print(f"\n⚖️ 啟動權重融合機制 (Alpha = {ALPHA_BASELINE})")
    print("-" * 50)
    # 進行 Alpha 凸組合融合 (Convex Combination)
    blended_weights = {}
    for key in BASELINE_WEIGHTS.keys():
        user_w = global_weights.get(key, 0.0)
        base_w = BASELINE_WEIGHTS[key]
        
        # 融合公式：W_final = α * W_base + (1 - α) * W_user
        blended_w = (ALPHA_BASELINE * base_w) + ((1 - ALPHA_BASELINE) * user_w)
        blended_weights[key] = blended_w
        
        print(f"{key:<15}: 使用者 {user_w*100:>5.2f}% | 融合後 -> {blended_w*100:>5.2f}%")
    print("-" * 50)
    # 將融合後的安全權重，指派給最佳化引擎使用的全局變數
    w_cagr = blended_weights["Return_CAGR"]
    w_div = blended_weights["Return_Div"]
    w_vol_risk = blended_weights["Risk_Vol"]
    w_maxdd = blended_weights["Risk_MaxDD"]
    w_cost = blended_weights["Cost_ExpRatio"]
    w_liq_vol = blended_weights["Liq_Volume"]
    w_liq_aum = blended_weights["Liq_AUM"]
    w_div_score = blended_weights["Div_Score"]
    w_sent = blended_weights["FinBERT_score"]

    # --- 🚨直接讀取已縮尾之正規化數據---
    vec_cagr = df_scaled_valid['Norm_Return_CAGR'].values
    vec_div = df_scaled_valid['Norm_Return_Div'].values
    vec_maxdd = df_scaled_valid['Norm_Risk_MaxDD'].values
    vec_cost = df_scaled_valid['Norm_Cost_ExpRatio'].values
    vec_liq_vol = df_scaled_valid['Norm_Liq_Volume'].values
    vec_liq_aum = df_scaled_valid['Norm_Liq_AUM'].values
    vec_div_score = df_scaled_valid['Norm_Div_Score'].values
    vec_sent = df_scaled_valid['Norm_FinBERT'].values

    # 共變異數矩陣動態正規化 (V_p = w^T * Sigma * w)
    cov_matrix_np = returns_matrix.cov().values * 252
    cov_matrix_norm = cov_matrix_np / np.max(cov_matrix_np)
    
    # ==========================================
    # 核心：定義全局效用函數 U(P)
    # ==========================================
    def calc_utility(w):
        """計算投資組合在當前 AHP 權重下的總效用分數 (Utility)"""
        port_cagr = np.dot(w, vec_cagr)
        port_div = np.dot(w, vec_div)
        port_maxdd = np.dot(w, vec_maxdd)
        port_cost = np.dot(w, vec_cost)
        port_liq_vol = np.dot(w, vec_liq_vol)
        port_liq_aum = np.dot(w, vec_liq_aum)
        # 🚨 分散度計算邏輯分歧
        if USE_TRUE_HHI_OPTIMIZATION and sector_matrix is not None:
            # 1. 算出投資組合在各產業的絕對總曝險 (1 x K 向量)
            port_sector_exposures = np.dot(w, sector_matrix)
            
            # 2. 計算真實投資組合 HHI (各產業權重的平方和)
            true_hhi = np.sum(port_sector_exposures ** 2)
            
            # 3. 轉換為 AHP 可理解的「正向效用」
            # HHI 範圍是 0 到 1 (越小越分散)。因此 1 - HHI = 真實分散度分數
            port_div_score= 1.0 - true_hhi 
        else:
            # 舊演算法：線性加權的代理指標
            port_div_score = np.dot(w, vec_div_score)
        port_sent = np.dot(w, vec_sent)
        
        port_vol_var = np.dot(w.T, np.dot(cov_matrix_norm, w))
        
        U = (w_cagr * port_cagr) + (w_div * port_div) \
            + (w_liq_vol * port_liq_vol) + (w_liq_aum * port_liq_aum) \
            + (w_div_score * port_div_score) + (w_sent * port_sent) \
            + (w_maxdd * port_maxdd) + (w_cost * port_cost) \
            - (w_vol_risk * port_vol_var)
        return U

    def objective_function(w):
        # SciPy 求最小，故回傳負效用
        return -calc_utility(w)

    # ... 執行偏好驅動最佳化 ...
    MAX_WEIGHT_LIMIT = 0.40 
    weight_bounds = tuple((0.0, MAX_WEIGHT_LIMIT) for _ in range(n_assets)) #上限為 0.40 (防止單一標的過度集中，最高 40%)
    bounds = tuple((0.0, 1.0) for _ in range(n_assets))
    constraints = ({'type': 'eq', 'fun': lambda w: np.sum(w) - 1.0})
    initial_w = np.array([1.0 / n_assets] * n_assets)
    
    result = minimize(objective_function, initial_w, method='SLSQP', bounds=weight_bounds, constraints=constraints, options={'maxiter': 1000, 'ftol': 1e-9})
    if not result.success:
        print("❌ 最佳化求解失敗：", result.message)
        return
    optimal_weights = np.round(result.x, 4)

    # ... 執行傳統 Max Sharpe 最佳化 ...
    annual_returns_array = returns_matrix.mean().values * 252
    cov_matrix_annual = returns_matrix.cov().values * 252

    def neg_sharpe_objective(w):
        p_ret = np.dot(w, annual_returns_array)
        p_vol = np.sqrt(np.dot(w.T, np.dot(cov_matrix_annual, w)))
        return - (p_ret - 0.04) / p_vol if p_vol > 0 else 0

    res_sharpe = minimize(neg_sharpe_objective, initial_w, method='SLSQP', bounds=bounds, constraints=constraints, options={'maxiter': 1000, 'ftol': 1e-9})
    max_sharpe_weights = np.round(res_sharpe.x, 4) if res_sharpe.success else initial_w
    if not res_sharpe.success:
        print(f"⚠️ 夏普組合求解未完全收斂: {res_sharpe.message}")

    # ==========================================
    # 🚨 新增：計算兩者的「偏好效用分數 U(P)」
    # ==========================================
    pref_utility_score = calc_utility(optimal_weights)
    ms_utility_score = calc_utility(max_sharpe_weights)


    # 呼叫視覺化函式
    plot_portfolio_analytics_and_mpt(returns_matrix, optimal_weights, max_sharpe_weights, valid_tickers)

    # ==========================================
    # 深度資料計算 (建立通用的計算函式)
    # ==========================================
    def get_portfolio_metrics(weights):
        # 計算每日與年化報酬、波動率、夏普
        port_daily = returns_matrix.dot(weights)
        volatility = port_daily.std() * np.sqrt(252)
        annual_ret = port_daily.mean() * 252
        sharpe = (annual_ret - 0.04) / volatility if volatility > 0 else 0
        
        # 計算最大回撤 (Max Drawdown)
        cum_ret = (1 + port_daily).cumprod()
        max_dd = ((cum_ret - cum_ret.cummax()) / cum_ret.cummax()).min() * 100
        
        # 計算線性加權的原始特徵指標
        exp_ratio = np.dot(weights, df_merged_valid['Cost_ExpRatio (%)'].fillna(0))
        cagr = np.dot(weights, df_merged_valid['Return_CAGR (%)'].fillna(0))
        div_yield = np.dot(weights, df_merged_valid['Return_Div (%)'].fillna(0))
        
        # 🚨 加入：計算線性加權的代理分散度 (確保 df_merged_valid 有 Div_Score 欄位)
        proxy_div = np.dot(weights, df_merged_valid['Div_Score (產出)'].fillna(0))

        return {
            'Arithmetic_Ret': annual_ret * 100,
            'CAGR': cagr,
            'Div_Yield': div_yield,
            'Cost': exp_ratio,
            'Volatility': volatility * 100,
            'MaxDD': max_dd,
            'Proxy_Div': proxy_div,
            'Sharpe': sharpe
        }

    # 分別取得兩組權重的深度數據
    pref_metrics = get_portfolio_metrics(optimal_weights)
    ms_metrics = get_portfolio_metrics(max_sharpe_weights)

    # 🚨 動態計算真實投資組合 HHI (若矩陣已成功建立)
    if sector_matrix is not None and len(sector_matrix) > 0:
        true_hhi_pref = np.sum(np.dot(optimal_weights, sector_matrix) ** 2)
        true_hhi_ms = np.sum(np.dot(max_sharpe_weights, sector_matrix) ** 2)
        hhi_str_pref = f"{true_hhi_pref:.4f}"
        hhi_str_ms = f"{true_hhi_ms:.4f}"
    else:
        hhi_str_pref = "N/A (API Limit)"
        hhi_str_ms = "N/A (API Limit)"

    # ==========================================
    # 輸出比較報表
    # ==========================================
    comparison_df = pd.DataFrame({
        'ETF': valid_tickers,
        '偏好組合 Weight (%)': optimal_weights * 100,
        '最大夏普 Weight (%)': max_sharpe_weights * 100
    })
    comparison_df = comparison_df[(comparison_df['偏好組合 Weight (%)'] > 0.01) | (comparison_df['最大夏普 Weight (%)'] > 0.01)]
    comparison_df = comparison_df.sort_values(by='偏好組合 Weight (%)', ascending=False).reset_index(drop=True)
    
    print("\n" + "="*65)
    print(" 🎯 專題最終產出：持股權重對比")
    print("="*65)
    print(comparison_df.to_string(index=False))
    
    # 建立深度健檢報告的 DataFrame 確保排版對齊
    analytics_df = pd.DataFrame({
        'Metric': [
            'Arithmetic Annual Return (%)', 
            'Historical CAGR (%)', 
            'Dividend Yield (%)', 
            'Expense Ratio (%)', 
            'Annualized Volatility (%)', 
            'Maximum Drawdown (%)', 
            'True Portfolio HHI (Real)',
            'Sharpe Ratio'
        ],
        'Preference-Driven': [
            f"{pref_metrics['Arithmetic_Ret']:.2f}", 
            f"{pref_metrics['CAGR']:.2f}", 
            f"{pref_metrics['Div_Yield']:.2f}", 
            f"{pref_metrics['Cost']:.3f}", 
            f"{pref_metrics['Volatility']:.2f}", 
            f"{pref_metrics['MaxDD']:.2f}",
            f"{hhi_str_pref}", 
            f"{pref_metrics['Sharpe']:.3f}"
        ],
        'Max Sharpe': [
            f"{ms_metrics['Arithmetic_Ret']:.2f}", 
            f"{ms_metrics['CAGR']:.2f}", 
            f"{ms_metrics['Div_Yield']:.2f}", 
            f"{ms_metrics['Cost']:.3f}", 
            f"{ms_metrics['Volatility']:.2f}", 
            f"{ms_metrics['MaxDD']:.2f}",
            f"{hhi_str_ms}",
            f"{ms_metrics['Sharpe']:.3f}"
        ]
    })

    print("\n" + "="*65)
    print(" 📊 投資組合深度健檢報告 (Portfolio Analytics Comparison)")
    print("="*65)
    print(analytics_df.to_string(index=False, line_width=65,justify='right'))
    print("-" * 65)
    
    # 輸出效用對決
    pref_utility_score = calc_utility(optimal_weights)
    ms_utility_score = calc_utility(max_sharpe_weights)
    print(f"   偏好驅動組合 【AHP 總效用分數】 : {pref_utility_score:.4f}  <-- 系統在此維度勝出！")
    print(f"   傳統最大夏普 【AHP 總效用分數】 : {ms_utility_score:.4f}")
    print("="*65)
if __name__ == "__main__":
    run_stage3_pipeline()

啟動 Stage 3: 偏好驅動二次規劃與投資組合深度分析...
⏳ 載入 17 檔 ETF 進行最佳化與歷史回測...

啟動本地快取引擎 (請求總數: 17 檔)...
⚡ 所有請求的 ETF 皆已在本地快取中，直接載入！

⚖️ 啟動權重融合機制 (Alpha = 0.5)
--------------------------------------------------
Return_CAGR    : 使用者 39.53% | 融合後 -> 27.26%
Return_Div     : 使用者  4.39% | 融合後 ->  4.70%
Risk_Vol       : 使用者  1.40% | 融合後 ->  5.70%
Risk_MaxDD     : 使用者  1.40% | 融合後 -> 10.70%
Cost_ExpRatio  : 使用者 21.15% | 融合後 -> 20.58%
Liq_Volume     : 使用者  1.37% | 融合後 ->  3.19%
Liq_AUM        : 使用者  4.12% | 融合後 ->  4.56%
Div_Score      : 使用者 21.15% | 融合後 -> 18.08%
FinBERT_score  : 使用者  5.49% | 融合後 ->  5.24%
--------------------------------------------------

啟動視覺化模組：計算歷史軌跡與 MPT 效率前緣...
✅ 產出圖表：png\test_portfolio_performance.png
✅ 產出圖表：png\test_mpt_efficient_frontier.png

 🎯 專題最終產出：持股權重對比
 ETF  偏好組合 Weight (%)  最大夏普 Weight (%)
 VOO             40.0             0.00
 VTV             40.0             4.92
SCHF             20.0            23.29
 VYM              0.0             6.20
SCHG              0.0            5